In [0]:
catalog = "meu_catalog"

silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Silver: {silver_schema}")
print(f"Gold: {gold_schema}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Conferência das tabelas Silver esperadas
display(
    spark.sql(f"SHOW TABLES IN {silver_schema}")
)

In [0]:

tb_info = spark.table(f"{silver_schema}.tb_info_filmes")
tb_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
tb_metrics = spark.table(f"{silver_schema}.tb_metricas_engajamento")
tb_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
tb_genres = spark.table(f"{silver_schema}.tb_generos")
tb_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")
tb_fx = spark.table(f"{silver_schema}.tb_cotacao_dolar")

for nome, df in {
    "tb_info_filmes": tb_info,
    "tb_financeiro_filmes": tb_fin,
    "tb_metricas_engajamento": tb_metrics,
    "tb_avaliacoes_usuarios": tb_reviews,
    "tb_generos": tb_genres,
    "tb_pessoas_empresas": tb_people,
    "tb_cotacao_dolar": tb_fx,
}.items():
    print(f"{nome}:")
    print(df.columns)

In [0]:
def dedup_por_filme(df):
    w = Window.partitionBy("id_filme").orderBy(F.col("id_filme"))
    return (
        df.withColumn("_rn_gold", F.row_number().over(w))
          .filter(F.col("_rn_gold") == 1)
          .drop("_rn_gold")
    )

info_1filme = dedup_por_filme(tb_info)
fin_1filme = dedup_por_filme(tb_fin)
metrics_1filme = dedup_por_filme(tb_metrics)

filmes_lancados = (
    info_1filme
    .filter(
        (F.col("status_filme") == "Lançado") &
        F.col("data_lancamento").isNotNull() &
        (F.col("data_lancamento") <= F.current_date())
    )
)

print(f"Filmes lançados considerados na fato: {filmes_lancados.count()}")

In [0]:

dim_movies = (
    info_1filme
    .select(
        F.xxhash64(F.col("id_filme")).alias("sk_movie_id"),
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse"),
    )
    .dropDuplicates(["id_filme"])
)

(
    dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_movies")
)

display(spark.table(f"{gold_schema}.dim_movies").limit(10))

In [0]:
genres_clean = (
    tb_genres
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_genero").cast("string")).alias("nome_genero")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_genero").isNotNull() &
        (F.col("nome_genero") != "")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

dim_genres = (
    genres_clean
    .select("nome_genero")
    .dropDuplicates()
    .withColumn("sk_genre_id", F.xxhash64(F.col("nome_genero")))
    .select("sk_genre_id", "nome_genero")
)

bridge_movie_genre = (
    genres_clean
    .join(dim_genres, on="nome_genero", how="inner")
    .join(
        dim_movies.select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

In [0]:
from pyspark.sql import functions as F

dim_movies_gold = spark.table(f"{gold_schema}.dim_movies")
dim_genres_gold = spark.table(f"{gold_schema}.dim_genres")
tb_genres_gold = spark.table(f"{silver_schema}.tb_generos")

genres_clean = (
    tb_genres_gold
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_genero").cast("string")).alias("nome_genero")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_genero").isNotNull() &
        (F.col("nome_genero") != "")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

bridge_movie_genre = (
    genres_clean.alias("g")
    .join(
        dim_movies_gold.select(
            "sk_movie_id",
            "id_filme"
        ).alias("m"),
        F.col("g.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .join(
        dim_genres_gold.select(
            "sk_genre_id",
            "nome_genero"
        ).alias("d"),
        F.col("g.nome_genero") == F.col("d.nome_genero"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id"),
        F.col("d.sk_genre_id")
    )
    .dropDuplicates()
)

print(f"Registros da bridge: {bridge_movie_genre.count():,}")

(
    bridge_movie_genre
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")
)

print("Tabela gold.bridge_movie_genre criada com sucesso.")

display(
    spark.table(f"{gold_schema}.bridge_movie_genre").limit(20)
)

In [0]:
people_clean = (
    tb_people
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.trim(F.col("nome_entidade").cast("string")).alias("nome_entidade"),
        F.trim(F.col("tipo_entidade").cast("string")).alias("tipo_entidade")
    )
    .filter(
        F.col("id_filme").isNotNull() &
        F.col("nome_entidade").isNotNull() &
        (F.col("nome_entidade") != "") &
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
    )
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

people_only = (
    people_clean.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select("nome_entidade", "tipo_entidade").dropDuplicates()
)

dim_people = (
    people_only
    .withColumn("sk_person_id", F.xxhash64(F.col("nome_entidade"), F.col("tipo_entidade")))
    .select("sk_person_id", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)

companies_only = people_clean.filter(F.col("tipo_entidade") == "Produtora").select("nome_entidade").dropDuplicates()

dim_companies = (
    companies_only.withColumn("sk_company_id", F.xxhash64(F.col("nome_entidade")))
    .select("sk_company_id", F.col("nome_entidade").alias("nome_produtora"))
)

for df, nome in [(dim_people, "dim_people"), (dim_companies, "dim_companies")]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{gold_schema}.{nome}"))

display(spark.table(f"{gold_schema}.dim_people").limit(20))
display(spark.table(f"{gold_schema}.dim_companies").limit(20))

In [0]:
bridge_movie_person = (
    people_clean.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        dim_people.select(
            "sk_person_id",
            F.col("nome_pessoa").alias("nome_entidade"),
            F.col("tipo_pessoa").alias("tipo_entidade")
        ),
        on=["nome_entidade", "tipo_entidade"], how="inner"
    )
    .select("sk_movie_id", "sk_person_id").dropDuplicates()
)

bridge_movie_company = (
    people_clean.filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        dim_companies.select("sk_company_id", F.col("nome_produtora").alias("nome_entidade")),
        on="nome_entidade", how="inner"
    )
    .select("sk_movie_id", "sk_company_id").dropDuplicates()
)

for df, nome in [(bridge_movie_person, "bridge_movie_person"), (bridge_movie_company, "bridge_movie_company")]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{gold_schema}.{nome}"))

display(spark.table(f"{gold_schema}.bridge_movie_person").limit(20))
display(spark.table(f"{gold_schema}.bridge_movie_company").limit(20))

In [0]:
reviews_by_movie = (
    tb_reviews.groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

dim_reviews = (
    reviews_by_movie
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .select(
        F.xxhash64(F.col("sk_movie_id")).alias("sk_review_id"),
        F.col("sk_movie_id"),
        F.col("qtd_avaliacoes_usuarios"),
        F.col("nota_media_usuarios")
    )
)

(dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.dim_reviews"))

display(spark.table(f"{gold_schema}.dim_reviews").limit(20))

In [0]:
fact_movies_performance = (
    filmes_lancados
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(
        fin_1filme.select("id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"),
        on="id_filme", how="left"
    )
    .join(
        metrics_1filme.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"),
        on="id_filme", how="left"
    )
    .select(
        F.col("sk_movie_id"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
    .dropDuplicates(["sk_movie_id"])
)

(fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.fact_movies_performance"))

display(spark.table(f"{gold_schema}.fact_movies_performance").limit(10))

In [0]:
atores = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), F.col("b.sk_person_id") == F.col("p.sk_person_id"), "inner")
    .filter(F.col("p.tipo_pessoa") == "Ator")
    .groupBy("b.sk_movie_id")
    .agg(F.concat_ws(", ", F.array_sort(F.collect_set(F.col("p.nome_pessoa")))).alias("atores_principais"))
)

diretores = (
    bridge_movie_person.alias("b")
    .join(dim_people.alias("p"), F.col("b.sk_person_id") == F.col("p.sk_person_id"), "inner")
    .filter(F.col("p.tipo_pessoa") == "Diretor")
    .groupBy("b.sk_movie_id")
    .agg(F.concat_ws(", ", F.array_sort(F.collect_set(F.col("p.nome_pessoa")))).alias("diretor"))
)

genai_context = (
    dim_movies.alias("m")
    .join(fact_movies_performance.alias("f"), on="sk_movie_id", how="left")
    .join(atores.alias("a"), on="sk_movie_id", how="left")
    .join(diretores.alias("d"), on="sk_movie_id", how="left")
    .select(
        F.col("m.id_filme").alias("movie_id"),
        F.col("m.titulo").alias("title"),
        F.concat(
            F.lit("O filme "),
            F.coalesce(F.col("m.titulo"), F.lit("Título não informado")),
            F.lit(", lançado no ano de "),
            F.coalesce(F.col("m.ano_lancamento").cast("string"), F.lit("ano não informado")),
            F.lit(", faturou "),
            F.coalesce(F.col("f.receita_usd").cast("string"), F.lit("receita não informada")),
            F.lit(" e teve um custo de "),
            F.coalesce(F.col("f.orcamento_usd").cast("string"), F.lit("orçamento não informado")),
            F.lit(". Estrelado por "),
            F.coalesce(F.col("a.atores_principais"), F.lit("elenco não informado")),
            F.lit(" e dirigido por "),
            F.coalesce(F.col("d.diretor"), F.lit("diretor não informado")),
            F.lit(", o filme possui a seguinte sinopse: "),
            F.coalesce(F.when(F.trim(F.col("m.sinopse")) != "", F.col("m.sinopse")), F.lit("Sinopse não informada.")),
            F.lit(".")
        ).alias("llm_context_document")
    )
)

(genai_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{gold_schema}.gold_genai_movies_context"))

display(spark.table(f"{gold_schema}.gold_genai_movies_context").limit(10))


In [0]:
tabelas_gold = [
    "fact_movies_performance", "dim_movies", "dim_genres", "dim_people",
    "dim_companies", "dim_reviews", "bridge_movie_genre",
    "bridge_movie_person", "bridge_movie_company", "gold_genai_movies_context"
]

for tabela in tabelas_gold:
    qtd = spark.table(f"{gold_schema}.{tabela}").count()
    print(f"{tabela}: {qtd:,} registros")

qtd_fato = spark.table(f"{gold_schema}.fact_movies_performance").count()
qtd_fato_distinto = spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id").distinct().count()

print(f"\nGrão da fato: {qtd_fato:,} linhas")
print(f"Filmes distintos na fato: {qtd_fato_distinto:,}")

if qtd_fato != qtd_fato_distinto:
    raise ValueError("ERRO: a fact_movies_performance possui mais de uma linha por filme.")

print("OK: a fato mantém um único registro por filme.")